[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aims-foundations/torch_measure/blob/main/tutorials/metr_agentic_evaluation.ipynb)

# METR Agentic Evaluation: IRT Analysis of AI Agent Capabilities

This tutorial demonstrates how to use **torch_measure** to analyze the [METR eval-analysis-public](https://github.com/METR/eval-analysis-public) dataset — a large-scale evaluation of AI agents on real-world tasks.

METR evaluated **34 AI agents** (Claude, GPT, Gemini, DeepSeek, Grok, etc.) on **170 tasks** spanning software engineering, machine learning, cybersecurity, and more. Each (agent, task) pair was evaluated across **multiple runs**, producing both binary pass/fail scores and continuous [0, 1] scores.

**What you'll learn:**
1. Loading the METR dataset from the `torch_measure.datasets` registry
2. Exploring the response matrix structure and metadata
3. Fitting **Beta IRT** models (BetaRasch, BetaTwoPL) for continuous pass-rate data
4. Comparing agent abilities and task difficulties across task sources (HCAST, RE-Bench, SWAA)
5. Evaluating model fit with train/test splitting and psychometric metrics
6. Visualizing item characteristic curves and ability rankings

## 1. Setup

In [ ]:
try:
    import google.colab
    !git clone https://github.com/aims-foundations/torch_measure.git
    !pip install -e "torch_measure[data]"
except ImportError:
    pass  # Already installed locally

import torch
import matplotlib.pyplot as plt
import numpy as np

from torch_measure.datasets import load, list_datasets, info
from torch_measure.data import ResponseMatrix, random_mask, binarize
from torch_measure.models import BetaRasch, BetaTwoPL, Rasch, TwoPL
from torch_measure.metrics import (
    expected_calibration_error, brier_score,
    cronbach_alpha, infit_statistics, outfit_statistics,
    ability_standard_errors, difficulty_standard_errors,
)

plt.rcParams["figure.dpi"] = 120
torch.manual_seed(42)
print("Setup complete.")

## 2. Explore Available METR Datasets

The METR data is organized into 8 datasets:
- **`metr/all`** and **`metr/all_score`**: All 170 tasks (pass rate and continuous score)
- **Per task source**: `metr/hcast`, `metr/rebench`, `metr/swaa` (and their `_score` variants)

All values are continuous [0, 1] because multiple runs per (agent, task) pair are averaged.

In [ ]:
print("Available METR datasets:")
for name in list_datasets(family="metr"):
    ds_info = info(name)
    print(f"  {name:25s}  {ds_info.n_subjects:>3} agents x {ds_info.n_items:>4} tasks  —  {ds_info.description}")

## 3. Load and Explore the Data

In [ ]:
# Load the main dataset: mean pass rate across runs
rm = load("metr/all")
print(rm)
print(f"Density (fraction observed): {rm.density:.1%}")
print(f"Overall mean pass rate: {rm.data[rm.observed_mask].mean():.3f}")
print(f"\nNumber of agents: {rm.n_subjects}")
print(f"Number of tasks: {rm.n_items}")

In [ ]:
# Inspect agents (subjects)
print("Agents and their mean pass rates:")
agent_means = rm.subject_means
ranking = agent_means.argsort(descending=True)
for rank, idx in enumerate(ranking):
    name = rm.subject_ids[idx]
    meta = rm.subject_metadata[idx] if rm.subject_metadata else {}
    org = meta.get("org", "")
    print(f"  {rank+1:2d}. {name:35s}  pass_rate={agent_means[idx]:.3f}  org={org}")

In [ ]:
# Inspect tasks (items) — show easiest and hardest
task_means = rm.item_means
task_ranking = task_means.argsort(descending=True)

print("Easiest 5 tasks (highest pass rate):")
for idx in task_ranking[:5]:
    print(f"  {rm.item_ids[idx]:50s}  pass_rate={task_means[idx]:.3f}")

print("\nHardest 5 tasks (lowest pass rate):")
for idx in task_ranking[-5:]:
    print(f"  {rm.item_ids[idx]:50s}  pass_rate={task_means[idx]:.3f}")

In [ ]:
# Visualize the response matrix sorted by agent ability and task difficulty
sorted_subj = agent_means.argsort()
sorted_item = task_means.argsort()
sorted_data = rm.data[sorted_subj][:, sorted_item]

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(sorted_data.numpy(), aspect="auto", cmap="RdYlGn", vmin=0, vmax=1, interpolation="nearest")
ax.set_xlabel("Tasks (sorted by difficulty, easy → hard)")
ax.set_ylabel("Agents (sorted by ability, low → high)")
ax.set_title("METR Response Matrix (green = high pass rate, red = low)")
plt.colorbar(im, ax=ax, shrink=0.8, label="Pass Rate")
plt.tight_layout()
plt.show()
print("The Guttman pattern is visible: capable agents (top) pass more tasks.")

## 4. Fitting Beta IRT Models

Since the METR data has continuous [0, 1] pass rates (averaged over multiple runs), we use **Beta IRT** models. These model the response probability as:

- **Beta-Rasch**: $\mu_{ij} = \sigma(\theta_i - b_j)$, with Beta likelihood parameterized by $(\mu, \phi)$
- **Beta-2PL**: $\mu_{ij} = \sigma(a_j(\theta_i - b_j))$, adding item discrimination $a_j$

where $\theta_i$ is agent ability, $b_j$ is task difficulty, and $\phi$ is the precision parameter.

In [ ]:
n_subjects, n_items = rm.n_subjects, rm.n_items

# Clamp values away from exact 0 and 1 for Beta likelihood
data = rm.data.clone()
eps = 1e-4
data = data.clamp(eps, 1 - eps)

# Fit Beta-Rasch
beta_rasch = BetaRasch(n_subjects=n_subjects, n_items=n_items, phi=10.0)
history_br = beta_rasch.fit(data, max_epochs=500, lr=0.01, verbose=True)
print(f"Beta-Rasch final loss: {history_br['losses'][-1]:.4f}")

# Fit Beta-2PL
beta_2pl = BetaTwoPL(n_subjects=n_subjects, n_items=n_items, phi=10.0)
history_b2 = beta_2pl.fit(data, max_epochs=500, lr=0.01, verbose=True)
print(f"Beta-2PL final loss: {history_b2['losses'][-1]:.4f}")

In [ ]:
# Compare training curves
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_br["losses"], label="Beta-Rasch (1PL)")
ax.plot(history_b2["losses"], label="Beta-2PL")
ax.set_xlabel("Epoch")
ax.set_ylabel("Beta NLL Loss")
ax.set_title("Training Loss Curves")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Agent Ability Rankings

The IRT ability parameter $\theta_i$ provides a principled ranking of agents that accounts for task difficulty — unlike raw pass rates which treat all tasks equally.

In [ ]:
ability = beta_2pl.ability.detach()
ability_ranking = ability.argsort(descending=True)

print(f"{'Rank':<5} {'Agent':<38} {'IRT Ability':>12} {'Pass Rate':>10}  {'Org'}")
print("-" * 80)
for rank, idx in enumerate(ability_ranking):
    name = rm.subject_ids[idx]
    meta = rm.subject_metadata[idx] if rm.subject_metadata else {}
    org = meta.get("org", "")
    print(f"{rank+1:<5d} {name:<38s} {ability[idx]:>12.3f} {agent_means[idx]:>10.3f}  {org}")

In [ ]:
# Compare IRT ability vs. raw pass rate
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(agent_means.numpy(), ability.numpy(), s=40, alpha=0.7)

# Label a few notable agents
for idx in list(ability_ranking[:3]) + list(ability_ranking[-3:]):
    ax.annotate(
        rm.subject_ids[idx],
        (agent_means[idx].item(), ability[idx].item()),
        fontsize=7, ha="left", va="bottom",
    )

ax.set_xlabel("Raw Mean Pass Rate")
ax.set_ylabel("IRT Ability (Beta-2PL)")
ax.set_title("IRT Ability vs. Raw Pass Rate")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
r = torch.corrcoef(torch.stack([agent_means, ability]))[0, 1]
print(f"Correlation: {r:.3f} — mostly monotone, but IRT reweights by task difficulty.")

## 6. Task Difficulty Analysis

IRT difficulty parameters $b_j$ provide a calibrated difficulty scale. The 2PL model also estimates discrimination $a_j$ — how sharply each task distinguishes between agents of different ability.

In [ ]:
difficulty = beta_2pl.difficulty.detach()
discrimination = beta_2pl.discrimination.detach()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Difficulty distribution
axes[0].hist(difficulty.numpy(), bins=25, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Task Difficulty (b)")
axes[0].set_ylabel("Count")
axes[0].set_title("Task Difficulty Distribution")
axes[0].grid(True, alpha=0.3)

# Discrimination distribution
axes[1].hist(discrimination.numpy(), bins=25, edgecolor="black", alpha=0.7, color="orange")
axes[1].set_xlabel("Task Discrimination (a)")
axes[1].set_ylabel("Count")
axes[1].set_title("Task Discrimination Distribution")
axes[1].axvline(x=1.0, color="red", linestyle="--", alpha=0.5, label="a=1 (Rasch)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Difficulty vs. empirical pass rate
axes[2].scatter(task_means.numpy(), difficulty.numpy(), alpha=0.5, s=15)
axes[2].set_xlabel("Empirical Pass Rate")
axes[2].set_ylabel("IRT Difficulty (b)")
axes[2].set_title("Difficulty vs. Empirical Pass Rate")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Show highest and lowest discrimination tasks
disc_ranking = discrimination.argsort(descending=True)

print("Top 5 most discriminating tasks (best at separating agent ability):")
for idx in disc_ranking[:5]:
    print(f"  a={discrimination[idx]:.2f}  b={difficulty[idx]:.2f}  {rm.item_ids[idx]}")

print("\nBottom 5 least discriminating tasks (weak signal for ability):")
for idx in disc_ranking[-5:]:
    print(f"  a={discrimination[idx]:.2f}  b={difficulty[idx]:.2f}  {rm.item_ids[idx]}")

## 7. Item Characteristic Curves

An ICC shows the probability of passing a task as a function of agent ability. Steeper curves indicate higher discrimination.

In [ ]:
theta_range = torch.linspace(-4, 4, 200)

# Pick tasks with varying difficulty and discrimination
task_indices = [
    disc_ranking[0].item(),     # highest discrimination
    disc_ranking[len(disc_ranking)//4].item(),  # 75th percentile
    disc_ranking[len(disc_ranking)//2].item(),  # median
    disc_ranking[-1].item(),    # lowest discrimination
]

fig, ax = plt.subplots(figsize=(8, 5))
for idx in task_indices:
    a_j = discrimination[idx].item()
    b_j = difficulty[idx].item()
    icc = torch.sigmoid(a_j * (theta_range - b_j))
    label = f"{rm.item_ids[idx][:35]}... (a={a_j:.2f}, b={b_j:.2f})"
    ax.plot(theta_range.numpy(), icc.numpy(), label=label, linewidth=2)

# Overlay actual agent abilities as rug marks
ax.scatter(ability.numpy(), [-0.02] * len(ability), marker="|", color="black", s=50, alpha=0.5, label="Agent abilities")

ax.set_xlabel("Agent Ability (\u03b8)")
ax.set_ylabel("P(pass)")
ax.set_title("Item Characteristic Curves (Beta-2PL)")
ax.legend(fontsize=7, loc="upper left")
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

## 8. Comparing Across Task Sources

METR tasks come from three sources: **HCAST** (136 tasks), **RE-Bench** (7 tasks), and **SWAA** (27 tasks). Let's compare how agents perform across these sources.

In [ ]:
# Load per-source datasets
rm_hcast = load("metr/hcast")
rm_rebench = load("metr/rebench")
rm_swaa = load("metr/swaa")

print(f"HCAST:    {rm_hcast.n_subjects} agents x {rm_hcast.n_items} tasks, mean pass rate = {rm_hcast.data[rm_hcast.observed_mask].mean():.3f}")
print(f"RE-Bench: {rm_rebench.n_subjects} agents x {rm_rebench.n_items} tasks, mean pass rate = {rm_rebench.data[rm_rebench.observed_mask].mean():.3f}")
print(f"SWAA:     {rm_swaa.n_subjects} agents x {rm_swaa.n_items} tasks, mean pass rate = {rm_swaa.data[rm_swaa.observed_mask].mean():.3f}")

In [ ]:
# Fit Beta-Rasch on each source and compare ability estimates
abilities_by_source = {}
for name, source_rm in [("HCAST", rm_hcast), ("RE-Bench", rm_rebench), ("SWAA", rm_swaa)]:
    source_data = source_rm.data.clone().clamp(eps, 1 - eps)
    model = BetaRasch(n_subjects=source_rm.n_subjects, n_items=source_rm.n_items, phi=10.0)
    model.fit(source_data, max_epochs=500, lr=0.01, verbose=False)
    abilities_by_source[name] = {
        "ability": model.ability.detach(),
        "subject_ids": source_rm.subject_ids,
    }
    print(f"{name}: fitted Beta-Rasch")

In [ ]:
# Cross-source ability correlations
# Find common agents across sources
sources = list(abilities_by_source.keys())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [(0, 1), (0, 2), (1, 2)]

for ax, (i, j) in zip(axes, pairs):
    src_a, src_b = sources[i], sources[j]
    ids_a = abilities_by_source[src_a]["subject_ids"]
    ids_b = abilities_by_source[src_b]["subject_ids"]
    common = sorted(set(ids_a) & set(ids_b))

    if len(common) < 3:
        ax.set_title(f"{src_a} vs {src_b}\n(too few common agents)")
        continue

    ab_a = torch.tensor([abilities_by_source[src_a]["ability"][ids_a.index(c)].item() for c in common])
    ab_b = torch.tensor([abilities_by_source[src_b]["ability"][ids_b.index(c)].item() for c in common])

    r = torch.corrcoef(torch.stack([ab_a, ab_b]))[0, 1].item()
    ax.scatter(ab_a.numpy(), ab_b.numpy(), s=30, alpha=0.7)
    ax.set_xlabel(f"{src_a} Ability")
    ax.set_ylabel(f"{src_b} Ability")
    ax.set_title(f"{src_a} vs {src_b} (r={r:.3f})")
    ax.grid(True, alpha=0.3)

plt.suptitle("Cross-Source Ability Consistency", fontsize=13)
plt.tight_layout()
plt.show()

## 9. Train/Test Evaluation

We hold out 20% of entries to evaluate how well the Beta-2PL model generalizes.

In [ ]:
torch.manual_seed(42)

observed = rm.observed_mask
train_mask, test_mask = random_mask(observed, train_frac=0.8)

print(f"Training entries: {train_mask.sum().item():,}")
print(f"Test entries:     {test_mask.sum().item():,}")

# Fit Beta-2PL on training data only
model_eval = BetaTwoPL(n_subjects=n_subjects, n_items=n_items, phi=10.0)
history_eval = model_eval.fit(data, mask=train_mask, max_epochs=500, lr=0.01, verbose=False)

# Evaluate on held-out entries
probs = model_eval.predict().detach()
ece = expected_calibration_error(probs, data, mask=test_mask)
bs = brier_score(probs, data, mask=test_mask)

print(f"\nTest set metrics:")
print(f"  Expected Calibration Error: {ece:.4f}")
print(f"  Brier Score: {bs:.4f}")

In [ ]:
# Calibration plot: predicted vs. actual pass rates in bins
test_probs = probs[test_mask]
test_actual = data[test_mask]

n_bins = 10
bin_edges = torch.linspace(0, 1, n_bins + 1)
bin_accs = []
bin_confs = []
bin_sizes = []

for i in range(n_bins):
    mask_bin = (test_probs >= bin_edges[i]) & (test_probs < bin_edges[i + 1])
    if mask_bin.sum() > 0:
        bin_accs.append(test_actual[mask_bin].mean().item())
        bin_confs.append(test_probs[mask_bin].mean().item())
        bin_sizes.append(mask_bin.sum().item())

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(bin_confs, bin_accs, width=0.08, alpha=0.6, label="Actual")
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")
ax.set_xlabel("Predicted Pass Rate")
ax.set_ylabel("Actual Pass Rate")
ax.set_title(f"Calibration Plot (ECE = {ece:.4f})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Uncertainty Quantification

Fisher information-based standard errors quantify how precisely each agent's ability is estimated.

In [ ]:
# Standard errors from the full Beta-2PL model
ability_full = beta_2pl.ability.detach()
difficulty_full = beta_2pl.difficulty.detach()

se_ability = ability_standard_errors(ability_full, difficulty_full)

# Plot ability estimates with 95% confidence intervals
sorted_idx = ability_full.argsort()
sorted_ab = ability_full[sorted_idx]
sorted_se = se_ability[sorted_idx]
sorted_names = [rm.subject_ids[i] for i in sorted_idx]

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(sorted_ab))
ax.barh(y_pos, sorted_ab.numpy(), xerr=(1.96 * sorted_se).numpy(), height=0.6,
        alpha=0.7, capsize=3, color="steelblue")
ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_names, fontsize=7)
ax.set_xlabel("IRT Ability (\u03b8) with 95% CI")
ax.set_title("Agent Ability Estimates with Uncertainty")
ax.axvline(x=0, color="k", linestyle="--", alpha=0.3)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

## 11. Comparison: Beta IRT vs. Standard IRT on Binarized Data

We can also binarize the pass rates (threshold at 0.5) and fit standard IRT models. This lets us compare whether the continuous Beta IRT captures more information.

In [ ]:
# Binarize: pass rate > 0.5 → 1, else → 0
binary_data = binarize(rm.data, threshold=0.5)
print(f"Binarized accuracy: {binary_data[~torch.isnan(binary_data)].mean():.3f}")

# Fit standard Rasch and 2PL on binary data
rasch_bin = Rasch(n_subjects=n_subjects, n_items=n_items)
twopl_bin = TwoPL(n_subjects=n_subjects, n_items=n_items)

rasch_bin.fit(binary_data, max_epochs=500, verbose=False)
twopl_bin.fit(binary_data, max_epochs=500, verbose=False)

# Compare ability estimates: Beta-2PL vs. standard 2PL
ability_beta = beta_2pl.ability.detach()
ability_bin = twopl_bin.ability.detach()

r = torch.corrcoef(torch.stack([ability_beta, ability_bin]))[0, 1]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(ability_bin.numpy(), ability_beta.numpy(), s=40, alpha=0.7)
for idx in list(ability_ranking[:3]):
    ax.annotate(rm.subject_ids[idx], (ability_bin[idx].item(), ability_beta[idx].item()),
                fontsize=7, ha="left", va="bottom")
ax.set_xlabel("Standard 2PL Ability (binarized data)")
ax.set_ylabel("Beta-2PL Ability (continuous data)")
ax.set_title(f"Ability Comparison: Beta-2PL vs. Standard 2PL (r={r:.3f})")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Correlation: {r:.3f}")
print("Beta-2PL retains graded information that binarization discards.")

## 12. Psychometric Quality: Reliability and Item Fit

We assess the quality of the METR evaluation using classical psychometric diagnostics on the binarized data.

In [ ]:
# Cronbach's alpha on binarized data
alpha = cronbach_alpha(binary_data)
print(f"Cronbach's alpha: {alpha:.3f}")
print(f"  (> 0.9 = excellent, 0.8-0.9 = good, 0.7-0.8 = acceptable)")

# Item fit statistics
probs_rasch_bin = rasch_bin.predict().detach()
infit = infit_statistics(probs_rasch_bin, binary_data)
outfit = outfit_statistics(probs_rasch_bin, binary_data)

print(f"\nInfit MNSQ  — mean: {infit.mean():.3f}, range: [{infit.min():.3f}, {infit.max():.3f}]")
print(f"Outfit MNSQ — mean: {outfit.mean():.3f}, range: [{outfit.min():.3f}, {outfit.max():.3f}]")

n_underfit = (infit > 1.3).sum().item()
n_overfit = (infit < 0.7).sum().item()
print(f"\nItems: {n_underfit} underfit (infit > 1.3), {n_overfit} overfit (infit < 0.7) out of {n_items}")

In [ ]:
# Item fit visualization
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(difficulty_full.numpy(), infit.numpy(), alpha=0.6, s=20)
ax.axhline(y=1.0, color="k", linestyle="--", alpha=0.5)
ax.axhline(y=1.3, color="r", linestyle="--", alpha=0.5, label="Underfit threshold")
ax.axhline(y=0.7, color="b", linestyle="--", alpha=0.5, label="Overfit threshold")
ax.set_xlabel("Task Difficulty (b)")
ax.set_ylabel("Infit MNSQ")
ax.set_title("Item Fit: Infit vs. Difficulty")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Wright Map (Person-Item Map)

A Wright map displays both agent abilities and task difficulties on the same logit scale, making it easy to see how well the tasks cover the ability range.

In [ ]:
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(10, 7), sharey=True,
                                         gridspec_kw={"width_ratios": [1, 1]})

# Left: agent ability distribution
y_vals = ability_full.numpy()
ax_left.barh(range(len(y_vals)), y_vals[ability_full.argsort().numpy()],
             height=0.7, color="steelblue", alpha=0.7)
ax_left.set_yticks(range(len(y_vals)))
ax_left.set_yticklabels([rm.subject_ids[i] for i in ability_full.argsort()], fontsize=6)
ax_left.set_xlabel("Ability (\u03b8)")
ax_left.set_title("Agents")
ax_left.grid(True, alpha=0.3, axis="x")
ax_left.invert_xaxis()

# Right: task difficulty histogram (horizontal)
bins = np.linspace(difficulty_full.min().item() - 0.5, difficulty_full.max().item() + 0.5, 20)
ax_right.hist(difficulty_full.numpy(), bins=bins, orientation="horizontal",
              color="coral", alpha=0.7, edgecolor="black")
ax_right.set_xlabel("Number of Tasks")
ax_right.set_title("Tasks")
ax_right.grid(True, alpha=0.3, axis="x")

plt.suptitle("Wright Map: Agent Abilities vs. Task Difficulties", fontsize=13)
plt.tight_layout()
plt.show()

## Summary

In this tutorial we:

1. **Loaded** the METR agentic evaluation dataset using `torch_measure.datasets.load("metr/all")`
2. **Explored** the response matrix: 34 agents x 170 tasks with continuous pass rates
3. **Fitted Beta IRT models** (BetaRasch, BetaTwoPL) designed for continuous [0, 1] responses
4. **Ranked agents** by IRT ability — a principled alternative to raw pass rates
5. **Analyzed task difficulty and discrimination** via the 2PL model
6. **Compared across task sources** (HCAST, RE-Bench, SWAA) to check ability consistency
7. **Evaluated generalization** with train/test splitting and calibration metrics
8. **Quantified uncertainty** with Fisher-based standard errors
9. **Compared Beta IRT vs. standard IRT** on binarized data
10. **Assessed psychometric quality** with Cronbach's alpha, infit/outfit statistics

### Key Takeaways

- **Beta IRT** is the natural choice for multi-run pass-rate data — it retains graded information that binarization discards
- **IRT ability** provides a difficulty-adjusted ranking of agents, unlike raw pass rates which treat all tasks equally
- **Task discrimination** reveals which tasks are most informative for distinguishing agent capabilities
- The METR dataset's high density (99%+) and diverse task range make it ideal for IRT analysis

### References

- METR (2025). *Measuring AI Ability to Complete Long Tasks.* [GitHub](https://github.com/METR/eval-analysis-public)
- Noel, Y. & Dauvier, B. (2007). *A Beta Item Response Model for Continuous Bounded Responses.* Applied Psychological Measurement.